# Experiment set: Module 2 (automatic path) and Module 3 on the 32 census passes

Root `motion_transfer_exp` (24-frame canonical cuts; the six pairs' root is never touched). `configs/experiment_set.yaml` is frozen (2026-10-19).
No human step: the proxies of record are copied, the control review is its own gate record, the anchor is selected by the rule of record,
the prompt comes from the caption by the fixed rule, the handoff verdict is the gate outcomes. A pair that fails a gate is a recorded outcome
(`reports/experiment/run_all.json`) and the next pair runs. Nothing here tunes anything.

Phase-1 session: cells 1-5 (Module 2). Phase-2 session (Wan runtime): cells 1, 2, 6 (Module 3).

In [ ]:
# Drive mount, the GitHub token from Drive (never printed, never persisted into the repo), clone-or-pull of the PRIVATE repo into /content/repo
# (the same path every other notebook uses); an anonymous clone of a private repo fails with "could not read Username".
import os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
GITHUB_URL = "https://github.com/jnheinrich451-eng/mixture-motion-transfer.git"; BRANCH = "main"; REPO_DIR = Path("/content/repo")

def _load_secret(path):
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            return line.split("=", 1)[1].strip().strip('"').strip("'")
    raise ValueError(f"no KEY=VALUE line in {path}")

os.environ["GH_TOKEN"] = _load_secret("/content/drive/MyDrive/secrets/gh_token.env")
os.environ["HF_TOKEN"] = _load_secret("/content/drive/MyDrive/secrets/hf_token.env")
subprocess.run(["git", "config", "--global", "credential.helper", "store --file /tmp/.git-credentials"], check=True)
Path("/tmp/.git-credentials").write_text("https://x-access-token:" + os.environ["GH_TOKEN"] + "@github.com" + chr(10)); os.chmod("/tmp/.git-credentials", 0o600)
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_URL, str(REPO_DIR)], check=True)
else:
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])
    if r.returncode != 0:
        print("WARN: git pull failed: the VM copy has local edits or diverged; commit/stash them or delete /content/repo and re-run")
%cd {REPO_DIR}

In [ ]:
!rm -r /content/repo

In [ ]:
!ls /content/data/videos; ls /content/data/videos/DAVIS 2>/dev/null | head -3


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer python scripts/render_thesis_ground_band.py --pair 000000000265.2.005/vehicle


In [ ]:
!cd /content/repo && git status --short | head -20


In [ ]:
!cd /content/repo && git checkout -- . && git clean -fd measurements && git pull --ff-only origin main


In [ ]:
# Phase-1 runtime (Modules 1-2): bootstrap (secrets, pins, D4RT, SAM2), the experiment videos, the selection instruments' extra deps
!bash scripts/bootstrap.sh
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='John00451/experiment-testset', repo_type='dataset', local_dir='/content/data/videos')"
!pip uninstall -y torchaudio -q; pip install -q --no-deps pyiqa; pip install -q git+https://github.com/microsoft/MoGe.git

In [ ]:
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --auto --proxy-from /content/drive/MyDrive/motion_transfer --redo-from r2 --redo-pairs 000000000362.2.005/human 000000000431.5.009/human camel/human horsejump-high/human


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/check_m2c_fill_occlusion.py


In [ ]:
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --auto --proxy-from /content/drive/MyDrive/motion_transfer --occlusion-excludes-fill


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer MT_DATA=/content/data python scripts/render_thesis_heading_plane.py --pair 000000000265.2.005/vehicle --out /content/drive/MyDrive/motion_transfer/viz/thesis/000000000265.2.005/heading_plane_265.2.005_vehicle.png


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/check_m2e_gauge_refusal.py --pairs 000000000362.2.005/human 000000000431.5.009/human camel/human horsejump-high/human


In [ ]:
!python scripts/render_thesis_heading_plane.py --pair 000000000265.2.005/vehicle --frame /content/drive/MyDrive/motion_transfer/viz/thesis/000000000265.2.005/frames/frame_000.png --mask /content/drive/MyDrive/motion_transfer/viz/thesis/000000000265.2.005/mask_bool.npy


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer python scripts/render_thesis_ground_band.py --pair 000000000265.2.005/vehicle --out /content/drive/MyDrive/motion_transfer/viz/thesis/000000000265.2.005/ground_band_265.2.005_vehicle.png


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer python scripts/check_m2b2_recentre_fixture.py


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/check_m2c_self_transfer.py --pairs breakdance-flare/human hockey/human kite-walk/human


In [ ]:
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --auto --proxy-from /content/drive/MyDrive/motion_transfer --redo-from r2


In [ ]:
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='John00451/experiment-testset', repo_type='dataset', local_dir='/content/data/videos')"


In [ ]:
# SAM 3D environments (once per VM). Objects (vehicles' R2) lives in its own micromamba env; Body (humans' R2) installs into the
# system python and needs detectron2 importable there: each part is checked on its own, so a VM with the Objects env but no detectron2 still gets Body.
!test -x /content/micromamba/envs/sam3d-objects/bin/python || bash scripts/setup_sam3d.sh objects
!python -c 'import detectron2' 2>/dev/null && echo 'detectron2 importable' || bash scripts/setup_sam3d.sh body
!python -c 'import detectron2' 2>/dev/null && echo 'Body environment OK' || echo 'Body environment NOT OK: read the FAILED lines above (and the log under motion_transfer*/envs/)'

In [ ]:
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --auto --proxy-from /content/drive/MyDrive/motion_transfer


In [ ]:
# what exists per pair (read-only)
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --status 2>&1 | grep -v '^\[all\] skip'

In [ ]:
%%bash scripts/setup_sam3d.sh body


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp MT_DATA=/content/data python scripts/place_proxy_on_clip.py --clip 000000000064.3.002 --category human --key 7b1379d4dfce7cad

In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp MT_DATA=/content/data python scripts/place_proxy_on_clip.py --clip camel --category human --key 7b1379d4dfce7cad


In [ ]:
!cd /content/repo && bash scripts/setup_sam3d.sh body 2>&1 | grep -A 30 "detectron2"


In [ ]:
!grep -n -i "error" /tmp/detectron2_pip.log | grep -v "subprocess-exited\|Failed building\|Failed to build\|See above" | head -30


In [ ]:
!grep -n -B 40 "subprocess-exited-with-error" /tmp/detectron2_pip.log | head -60


In [ ]:
!cd /tmp && pip install -v --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' > /tmp/d2_verbose.log 2>&1; grep -n -i "error\|No module\|not found\|nvcc\|ninja" /tmp/d2_verbose.log | head -40


In [ ]:
!tail -n 60 /tmp/d2_verbose.log


In [ ]:
!nvcc --version | tail -1; python -c "import setuptools, wheel; print(setuptools.__version__, wheel.__version__)"


In [ ]:
!CUDA_VISIBLE_DEVICES="" CUDA_HOME=/nonexistent FORCE_CUDA=0 pip install -q ninja 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps && python -c "import detectron2; print('detectron2 importable')"


In [ ]:
!mkdir -p measurements/experiment && cp /content/drive/MyDrive/motion_transfer_exp/reports/experiment/run_all.json measurements/experiment/ && cp /content/drive/MyDrive/motion_transfer_exp/reports/m2_selection/selection.* measurements/experiment/ && ls measurements/experiment


In [ ]:
!bash scripts/bootstrap_phase2.sh
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 3 --auto



In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/collect_m3_set_records.py


In [ ]:
!cd /content/repo && git pull && python scripts/run_m3_ablation_tg.py --windows 0.85,0.75 0.85,0.45



In [ ]:
!cd /content/repo && git pull && python scripts/run_m3_ablation_tg.py --windows 0.85,0.75 0.85,0.45


In [ ]:
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 3 --auto


In [ ]:
!cp /content/drive/MyDrive/motion_transfer_exp/reports/experiment/run_all.json measurements/experiment/


In [ ]:
# Module 2 on every pair of the set, automatic path; resumable (a pair whose outputs exist is skipped stage by stage)
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 2 --auto --proxy-from /content/drive/MyDrive/motion_transfer
# the set-level selection report (CPU) once every pair has been through selection
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/select_m2d_anchors.py select --pairs $(python -c "import yaml; print(' '.join(p['pair'] for p in yaml.safe_load(open('configs/experiment_set.yaml'))['experiment_set']['pairs']))")
!mkdir -p measurements/experiment && cp /content/drive/MyDrive/motion_transfer_exp/reports/experiment/run_all.json measurements/experiment/ && cp /content/drive/MyDrive/motion_transfer_exp/reports/m2_selection/selection.* measurements/experiment/ && ls measurements/experiment

In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer python scripts/render_thesis_ground_band.py --pair 000000000265.2.005/vehicle


In [ ]:
# Phase-2 session only (scripts/bootstrap_phase2.sh first): Module 3 batch on every pair with a handoff; 21 frames, seed 0
!python scripts/run_all.py --env colab_exp --set configs/experiment_set.yaml --module 3 --auto
!cp /content/drive/MyDrive/motion_transfer_exp/reports/experiment/run_all.json measurements/experiment/

In [ ]:
!python -c "import cv2; [print(p, int(cv2.VideoCapture(p).get(cv2.CAP_PROP_FRAME_COUNT))) for p in ('/content/data/videos/000000000191.6.037.mp4','/content/data/videos/miradata/000000000191.6.037.mp4')]"


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/collect_m3_set_records.py


In [ ]:
!cd /content/repo && git pull && python scripts/record_experiment_video_hashes.py --videos-dir /content/data/videos


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/check_m2e_body_integrity.py


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/check_m3_bridge_interval.py --camera ba --pairs $(python -c "import yaml; print(' '.join(yaml.safe_load(open('configs/experiment_handoffs.yaml'))['experiment_handoffs']['main']))")
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/collect_m3_set_records.py


In [ ]:
!cp /content/repo/configs/experiment_video_hashes.yaml /content/drive/MyDrive/motion_transfer_exp/reports/experiment/


In [ ]:
!cd /content/repo && git pull && MT_ROOT=/content/drive/MyDrive/motion_transfer_exp MT_DATA=/content/data python scripts/report_experiment_source_geometry.py


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/collect_m3_set_records.py
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp MT_DATA=/content/data python scripts/run_m3_batch.py export --config configs/m3_batch_exp.yaml


In [ ]:
!cd /content/repo && git pull


In [ ]:
%%bash
MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/collect_m3_set_records.py
MT_ROOT=/content/drive/MyDrive/motion_transfer_exp MT_DATA=/content/data python scripts/run_m3_batch.py export --config configs/m3_batch_exp.yaml
